# Predicting Amazon Ratings Through Latent Factors and Feature-based Bias Terms

##### Names: Aiden Lai, Jordi Pham, Adrian Laksana, and Coltrane Gowan

##### Link To Slides: https://docs.google.com/presentation/d/1v6GXpv55cyt4lk9kVa9VeK0d0hDh_W08-coctd5C9_8/edit?usp=sharing

## Library And Data Set Up

In [ ]:
# Any Library used in this assignment should be imported here
import gzip
import csv
import random
import numpy as np
import math
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
# File opener
def open_file(file_path):
    data_grabber = []
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            data_grabber.append(json.loads(line))
    return data_grabber

In [3]:
# File path
mtv_fp = "./data/Movies_and_TV.jsonl"

## I. Our Predictive Task

The predictive task that we have settled on is predicting the ratings of the movies and tv subcategory of Amazon products: 

The foundational model we plan to use to fulfil this task is a latent factor model, and we plan to layer on additional factors for complexity such as terms that account for biases that can stem from features like 'helpful_vote' and 'verified_purchase' - that way, we are no longer just dealing with a latent factor model in its most basic form. In order to properly evaluate this model, we will work through effective data splits for test and train sets and measure performance through metrics like MSE and MAE.

There are two main baseline models that we can utilize throughout this predictive tasks. First is a model that simply uses user and global averages. This is a model that assumes a user's rating is equal to their historical rating, and when it encounters cold-start users, or users with no history, the model simply plugs in the global average. The second baseline model is a regularized bias-only model (this is is essentially the latent factor model without the interaction term between user and items). This model will serve effective in showing how useful it is to include the interaction term between user and items.

To test for validity, there will be a couple main things for us to focus on, all stemming from our initial design choices. Most importantly, we aim for the implementation of latent factors to increase the performance of the model in comparison to its regularized-bias-only counterpart. Furthermore, with our inclusion of features like 'helpful_vote' and 'verified_purchase', we aim to show that movie and tv ratings definitely have some bias rooted in other users' reviews.

Essentially, by the end of this exploration, we will have created these models:

1. User/Global Averages Model

2. Regularized Bias-Only Model

3. Latent Factor Model

4. Latent Factor + Additional Bias terms Model

## II. Exploratory Analysis, Data Collection, Pre-processing, and General Discussion

### Context:

The dataset that we've employed to construct and evaluate our models is the movies and tv category dataset. 

This dataset belongs to a much larger dataset of Amazon reviews product metadata, all collected and curated by the McAuley Lab at UCSD. The original purpose of the data was to pre-train BLAIR, a series of trained sentence-embedding models targetting recommendation scenarios, all designed by the lab to learn and recognize patterns between metadata and natural language context. Overall, the Amazon product review dataset (2023 version) is a dataset between May 1996 and September 2023 and contains 570 million reviews and 48 million items spanning across 33 categories with the movies and t category alone containing 6.5 million users and 747.8 thousand items.

### Discussion

The original Amazon product review dataset was processed for BLAIR by first, constructing language context through concatenating the title and context of user reviews, and then, constructing item metadata thought concateninating its respective title, feature, and descriptions. To mainintain quality throughout the model, the lab further set a 30 character threshold for the language context or item metadata.

In regards to our specific predictive task, we won't be diving into the construction of language context or concise item metadata. We are simply checking for feature types and extracting the exact features that we need.

### Code: Data Insights

In [ ]:
# Opening file
mtv_original = open_file(mtv_fp)

In [5]:
# Universal length variable
mtv_len = len(mtv_original)
mtv_len

17328314

In [6]:
# Raw Data View - Post Read
mtv_original[0]

{'rating': 5.0,
 'title': 'Five Stars',
 'text': "Amazon, please buy the show! I'm hooked!",
 'images': [],
 'asin': 'B013488XFS',
 'parent_asin': 'B013488XFS',
 'user_id': 'AGGZ357AO26RQZVRLGU4D4N52DZQ',
 'timestamp': 1440385637000,
 'helpful_vote': 0,
 'verified_purchase': True}

In [ ]:
# Viewable Dataframe Header
mtv_sample = mtv_original[0:5]
mtv_sample

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,"Amazon, please buy the show! I'm hooked!",[],B013488XFS,B013488XFS,AGGZ357AO26RQZVRLGU4D4N52DZQ,1440385637000,0,True
1,5.0,Five Stars,My Kiddos LOVE this show!!,[],B00CB6VTDS,B00CB6VTDS,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1461100610000,0,True
2,3.0,Some decent moments...but...,Annabella Sciorra did her character justice wi...,[],B096Z8Z3R6,B096Z8Z3R6,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1646271834582,0,True
3,4.0,"Decent Depiction of Lower-Functioning Autism, ...",...there should be more of a range of characte...,[],B09M14D9FZ,B09M14D9FZ,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1645937761864,1,False
4,5.0,What Love Is...,"...isn't always how you expect it to be, but w...",[],B001H1SVZC,B001H1SVZC,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1590639227074,0,True


In [8]:
# Columns
list(mtv_df.columns)

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

## III. Data Modeling

### Context

### Discussion

### Code: Architectural Choices and Implementation

In [ ]:

# Baseline model: user mean with global fallback

def clean_ratings(records):
    cleaned = []
    for record in records:
        try:
            user = record['user_id']
            item = record['parent_asin']
            rating = float(record['rating'])
        except (KeyError, TypeError, ValueError):
            continue
        cleaned.append((user, item, rating))
    return cleaned

ratings_clean = clean_ratings(mtv_original)

# adjust training splits later
train_set, test_set = train_test_split(ratings_clean, test_size=0.20, random_state=42)
train_set, val_set = train_test_split(train_set, test_size=0.125, random_state=42)  # 0.125 of 0.8 -> 0.1 overall

def compute_user_means(train_data):
    sums = {}
    counts = {}
    for user, _, rating in train_data:
        sums[user] = sums.get(user, 0.0) + rating
        counts[user] = counts.get(user, 0) + 1
    return {user: sums[user] / counts[user] for user in sums}

global_mean = float(np.mean([rating for _, _, rating in train_set])) if train_set else 0.0
user_means = compute_user_means(train_set)

def predict_user_global(records):
    return [user_means.get(user, global_mean) for user, _, _ in records]

def summarize_split(records, split_name):
    preds = predict_user_global(records)
    actuals = [rating for _, _, rating in records]
    mse = float(np.mean([(a - p) ** 2 for a, p in zip(actuals, preds)])) if actuals else 0.0
    mae = float(np.mean([abs(a - p) for a, p in zip(actuals, preds)])) if actuals else 0.0
    cold_start_users = 1.0 - float(np.mean([user in user_means for user, _, _ in records])) if records else 0.0
    return {
        'split': split_name,
        'mse': mse,
        'mae': mae,
        'cold_start_user_fraction': cold_start_users,
        'n': int(len(records)),
    }

In [ ]:
# Baseline model: regularized bias-only model

In [ ]:
# Original Latent Factor Model

In [ ]:
# Model with Latent Factor with Additional Feature-based Bias Terms (timestamps, purchase verification, and votes on the helpfulness of reviews)

## IV. Task Evaluation

### Context

### Discussion

### Code

In [ ]:
# Dataframe to save and compare performance metrics
performance_metrics = 

In [ ]:
baseline_results = [
    summarize_split(train_set, 'train'),
    summarize_split(val_set, 'validation'),
    summarize_split(test_set, 'test'),
]

import pprint
pprint.pprint(baseline_results)

## V. Related Works

Describe related-works topics here

# VI. Citations

Bridging Language and Items for Retrieval and Recommendation
Yupeng Hou, Jiacheng Li, Zhankui He, An Yan, Xiusi Chen, Julian McAuley
arXiv